In [28]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression,ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error,root_mean_squared_error,confusion_matrix,roc_auc_score,log_loss
from sklearn.metrics import f1_score,accuracy_score,recall_score,precision_score,classification_report
from tqdm import tqdm
import os
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler,PolynomialFeatures
from sklearn.neighbors import KNeighborsClassifier,KNeighborsRegressor
import datetime as dt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis,QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import BernoulliNB,GaussianNB
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier,plot_tree

In [29]:
cancer=pd.read_csv(r"../Datasets/BreastCancer.csv")
cancer

,Code,Clump,UniCell_Size,Uni_CellShape,MargAdh,SEpith,BareN,BChromatin,NoemN,Mitoses,Class
0,61634,5,4,3,1,2,2,2,3,1,Benign
1,63375,9,1,2,6,4,10,7,7,2,Malignant
2,76389,10,4,7,2,2,8,6,1,1,Malignant
3,95719,6,10,10,10,8,10,7,10,7,Malignant
4,128059,1,1,1,1,2,5,5,1,1,Benign
...,...,...,...,...,...,...,...,...,...,...,...
694,1369821,10,10,10,10,5,10,10,10,7,Malignant
695,1371026,5,10,10,10,4,10,5,6,3,Malignant
696,1371920,5,1,1,1,2,1,3,2,1,Benign
697,8233704,4,1,1,1,1,1,2,1,1,Benign


### SHORTENING OF TREE

In [30]:
X = cancer.drop(['Code','Class'],axis=1)
y=cancer['Class']
X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.3,random_state=26,stratify = cancer['Class'])

In [ ]:
dtc=DecisionTreeClassifier(random_state=26)
dtc.fit(X_train,y_train)
plt.figure(figsize=(35,25))
plt.xlabel("x1")
plt.ylabel("x2")
plot_tree(dtc,feature_names=list(X.columns),class_names=['absent','present'],filled=True,fontsize=12)
plt.show()

In [ ]:
y_pred=dtc.predict(X_test)
print("accuracy score:",accuracy_score(y_test,y_pred))
y_pred_prob=dtc.predict_proba(X_test)
print("log loss:",log_loss(y_test,y_pred_prob))

In [ ]:
dtc=DecisionTreeClassifier(random_state=26,max_depth=3)
dtc.fit(X_train,y_train)
plt.figure(figsize=(25,10))
plt.xlabel("x1")
plt.ylabel("x2")
plot_tree(dtc,feature_names=list(X.columns),class_names=['BENIGN','MALIGNANT'],filled=True,fontsize=12)
plt.show()

y_pred=dtc.predict(X_test)
print("accuracy score:",accuracy_score(y_test,y_pred))
y_pred_prob=dtc.predict_proba(X_test)
print("log loss:",log_loss(y_test,y_pred_prob))

### FOR DIFF LEAVES

In [ ]:
leaves = [2,5,10,20]
scores=[]
for l in leaves:
    dtc=DecisionTreeClassifier(random_state=26,min_samples_leaf=l)
    dtc.fit(X_train,y_train)
    y_pred_prob=dtc.predict_proba(X_test)
    scores.append([l,log_loss(y_test,y_pred_prob),accuracy_score(y_test,y_pred)])


df_scores = pd.DataFrame(scores,columns=['leaves','score','accuracy_score'])
df_scores.sort_values('score')

In [ ]:
depths=np.arange(1,7)
splits = np.arange(2,25)
leaves = np.arange(1,25)
scores=[]
for d in tqdm(depths):
    for s in splits:
        for l in leaves:
            dtc=DecisionTreeClassifier(random_state=26,max_depth=d,min_samples_split=s,min_samples_leaf=l)
            dtc.fit(X_train,y_train)
            y_pred_prob=dtc.predict_proba(X_test)
            y_pred=dtc.predict(X_test)
            scores.append([d,s,l,log_loss(y_test,y_pred_prob),accuracy_score(y_test,y_pred)])


df_scores = pd.DataFrame(scores,columns=['depths','splits','leaves','accuracy_score','score'])
df_scores.sort_values('score')

In [ ]:
bm = DecisionTreeClassifier(  
                random_state=26,
                min_samples_leaf=30,
                min_samples_split=2,
                max_depth=2
            )
bm.fit(X,y)
plt.figure(figsize=(25, 10))
plot_tree(bm, feature_names = list(X.columns), class_names = ["BENIGN", "MALIGNANT"], filled = True, fontsize = 12)
plt.show()

df_imp = pd.DataFrame({'feature':X.columns,'importance':bm.feature_importances_})
plt.barh(df_imp['feature'],df_imp['importance'])
plt.title('Feature Importance Plot')
plt.show()